# PhoMT vi→en Transformer (Pre-LN) — 600k mẫu, tokenizer mới, train từ đầu

In [1]:
# Cell 1 — Thiết lập Kaggle API + hàm tự động đẩy checkpoint ra dataset permanent
import os, json, subprocess, shutil

# 1. Ghi file xác thực đúng định dạng kaggle.json cần
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump({"username": "maihongsn", "key": "KGAT_51afb046245ae60a71a3e3300440d667"}, f)
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)

# 2. Cài/ nâng cấp kaggle CLI cho chắc
subprocess.run(["pip", "install", "-q", "--upgrade", "kaggle"])

# 3. Cấu hình dataset checkpoint sẽ dùng
DATASET_SLUG = "transformer-phomt-600k-ckpt" 
KAGGLE_USERNAME = "maihongsn"
UPLOAD_DIR = "/kaggle/working/ckpt_upload"
os.makedirs(UPLOAD_DIR, exist_ok=True)

TOKENIZER_PATH = "/kaggle/working/tokenizer/phomt_bpe.json"

def push_files_to_kaggle(file_paths, message="auto checkpoint"):
    """Copy danh sách file (checkpoint(s), tokenizer,...) vào dataset và đẩy lên Kaggle."""
    for p in file_paths:
        if p and os.path.exists(p):
            shutil.copy(p, os.path.join(UPLOAD_DIR, os.path.basename(p)))
    meta = {
        "title": DATASET_SLUG,
        "id": f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
        "licenses": [{"name": "CC0-1.0"}]
    }
    with open(f"{UPLOAD_DIR}/dataset-metadata.json", "w") as f:
        json.dump(meta, f)

    check = subprocess.run(["kaggle", "datasets", "status", meta["id"]], capture_output=True, text=True)
    if check.returncode != 0:
        r = subprocess.run(["kaggle", "datasets", "create", "-p", UPLOAD_DIR, "-q"], capture_output=True, text=True)
        print("TẠO MỚI dataset checkpoint:", r.stdout, r.stderr)
    else:
        r = subprocess.run(["kaggle", "datasets", "version", "-p", UPLOAD_DIR, "-m", message, "-q"],
                            capture_output=True, text=True)
        print("CẬP NHẬT checkpoint:", r.stdout, r.stderr)

# giữ hàm cũ để tương thích ngược (chỉ push 1 checkpoint + tokenizer)
def push_checkpoint_to_kaggle(ckpt_path, tokenizer_path=None, message="auto checkpoint"):
    push_files_to_kaggle([ckpt_path, tokenizer_path], message=message)

# 4. Kiểm tra token hoạt động đúng trước khi chạy tiếp
test = subprocess.run(["kaggle", "datasets", "list", "-m"], capture_output=True, text=True)
print("Kiểm tra Kaggle API:", "OK" if test.returncode == 0 else "LỖI")
print(test.stdout[:300], test.stderr[:300])

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 12.8 MB/s eta 0:00:00
Kiểm tra Kaggle API: OK
ref                                                title                                          size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-------------------------------------------------  ---------------------------------------  ----------  -------------------- 


In [2]:
import glob, os
from datasets import load_dataset, load_from_disk

candidates = glob.glob('/kaggle/input/**/dataset_dict.json', recursive=True)
if candidates:
    raw = load_from_disk(os.path.dirname(candidates[0]))
    print("Đã nạp PhoMT từ cache Kaggle:", os.path.dirname(candidates[0]))
else:
    raw = load_dataset("ura-hcmut/PhoMT")
    print("Không thấy cache, tải lại từ HuggingFace.")

Đã nạp PhoMT từ cache Kaggle: /kaggle/input/datasets/maihongsn/phomt-local-cache


In [3]:
# Lấy ~600k cặp câu từ ~3M của PhoMT + lọc rác
import numpy as np

N_TRAIN, N_VALID = 600_000, 5_000
SEED = 42

def pair_ok(v, e):
    if not (isinstance(v, str) and isinstance(e, str)):
        return False
    v, e = v.strip(), e.strip()
    if not v or not e or len(v) > 400 or len(e) > 400:
        return False
    r = len(v.split()) / max(1, len(e.split()))     # tiếng Việt ~1.3x số từ tiếng Anh
    return 0.5 <= r <= 3.0

def sample_and_clean(split, n, seed):
    rng = np.random.RandomState(seed)
    take = min(len(split), int(n * 1.2))              # lấy dư để bù phần bị lọc
    idx = np.sort(rng.permutation(len(split))[:take]) # sort để đọc đĩa nhanh hơn
    cand = split.select(idx.tolist())
    vis, ens = cand["vi"], cand["en"]
    good = [i for i, (v, e) in enumerate(zip(vis, ens)) if pair_ok(v, e)][:n]
    print(f"lấy {len(cand)} -> giữ {len(good)} cặp")
    return cand.select(good)

VALID_RAW = next(raw[k] for k in ("validation", "valid", "dev") if k in raw)
train_sub = sample_and_clean(raw["train"], N_TRAIN, SEED)
valid_sub = sample_and_clean(VALID_RAW, N_VALID, SEED)

train_vi, train_en = train_sub["vi"], train_sub["en"]
valid_vi, valid_en = valid_sub["vi"], valid_sub["en"]
print("train:", len(train_vi), "| valid:", len(valid_vi))
print("VI:", train_vi[0]); print("EN:", train_en[0])

lấy 720000 -> giữ 600000 cặp
lấy 6000 -> giữ 5000 cặp
train: 600000 | valid: 5000
VI: Ngày 14, tháng 8, năm 1947, gần nửa đêm, ở Bombay, có một phụ nữ sắp lâm bồn.
EN: On August 14th, 1947, a woman in Bombay goes into labor as the clock ticks towards midnight.


In [4]:
# Train BPE tokenizer MỚI (Metaspace) để decode giữ được dấu cách
import os
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, normalizers, Regex

TOK_VOCAB = 24000
os.makedirs(os.path.dirname(TOKENIZER_PATH), exist_ok=True)

tok = Tokenizer(models.BPE(unk_token="<unk>"))
tok.normalizer = normalizers.Sequence([
    normalizers.NFKC(),
    normalizers.Replace(Regex(r"\s+"), " "),
    normalizers.Strip(),
])

tok.pre_tokenizer = pre_tokenizers.Metaspace(replacement="\u2581", prepend_scheme="always")
tok.decoder = decoders.Metaspace(replacement="\u2581", prepend_scheme="always")

trainer = trainers.BpeTrainer(
    vocab_size=TOK_VOCAB, min_frequency=2, limit_alphabet=2000, show_progress=True,
    special_tokens=["<pad>", "<unk>", "<s>", "</s>"],
)

def corpus(bs=10000):
    for i in range(0, len(train_vi), bs):
        yield train_vi[i:i + bs]
        yield train_en[i:i + bs]

tok.train_from_iterator(corpus(), trainer=trainer)
tok.save(TOKENIZER_PATH)
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

PAD, UNK, BOS, EOS = 0, 1, 2, 3
assert [tokenizer.token_to_id(t) for t in ("<pad>", "<unk>", "<s>", "</s>")] == [0, 1, 2, 3]
VOCAB_SIZE = tokenizer.get_vocab_size()

class SPWrapper:
    def __init__(self, tok):
        self.tok = tok
    def encode(self, text, out_type=int):
        return self.tok.encode(text).ids
    def decode(self, ids):
        return self.tok.decode(ids)
    def get_piece_size(self):
        return self.tok.get_vocab_size()

sp = SPWrapper(tokenizer)
print("vocab size:", VOCAB_SIZE)

# Kiểm tra round-trip sẽ thấy khoảng trắng được phục hồi hoàn hảo
tests = ["Sadly, Brother Albert Barnett and his wife, Susan, were killed.",
         "Hôm nay trời đẹp, chúng ta đi chơi nhé!",
         "Tôi đang học dịch máy tại HCMUS."]
for s in tests:
    ids = tokenizer.encode(s).ids
    back = tokenizer.decode(ids)
    print(("OK  " if back == s else "LỆCH"), tokenizer.encode(s).tokens[:10])
    print("    ", repr(back))

tok.model.save(os.path.dirname(TOKENIZER_PATH))     # ghi vocab.json + merges.txt riêng
from tokenizers.models import BPE as _BPE

tokenizer_train = Tokenizer.from_file(TOKENIZER_PATH)
tokenizer_train.model = _BPE.from_file(
    f"{os.path.dirname(TOKENIZER_PATH)}/vocab.json",
    f"{os.path.dirname(TOKENIZER_PATH)}/merges.txt",
    dropout=0.1, unk_token="<unk>",
)




vocab size: 24000
OK   ['▁Sad', 'ly,', '▁Brother', '▁Albert', '▁Barn', 'ett', '▁and', '▁his', '▁wife,', '▁Sus']
     'Sadly, Brother Albert Barnett and his wife, Susan, were killed.'
OK   ['▁Hôm', '▁nay', '▁trời', '▁đẹp,', '▁chúng', '▁ta', '▁đi', '▁chơi', '▁nhé!']
     'Hôm nay trời đẹp, chúng ta đi chơi nhé!'
OK   ['▁Tôi', '▁đang', '▁học', '▁dịch', '▁máy', '▁tại', '▁H', 'C', 'M', 'U']
     'Tôi đang học dịch máy tại HCMUS.'


In [5]:
import math, torch
import torch.nn as nn
import torch.nn.functional as F

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)          # thêm dropout sau khi cộng pe, đúng chuẩn paper gốc
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=512, n_heads=8, n_layers=6,
                 d_ff=2048, dropout=0.1, max_len=256, pad_id=0, tie_weights=True):
        super().__init__()
        self.pad_id = pad_id
        self.src_emb = nn.Embedding(src_vocab, d_model, padding_idx=pad_id)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model, padding_idx=pad_id)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout=dropout)

        # Gọi thẳng Transformer của Torch. norm_first=True tương ứng kiến trúc Pre-LN
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=n_heads,
            num_encoder_layers=n_layers,
            num_decoder_layers=n_layers,
            dim_feedforward=d_ff,
            dropout=dropout,
            activation="relu",
            batch_first=True,
            norm_first=True 
        )
        self.out_proj = nn.Linear(d_model, tgt_vocab)
        self.scale = math.sqrt(d_model)
        self._init_weights(d_model, pad_id)

        # Tie weight: vi và en dùng CHUNG 1 tokenizer (src_vocab == tgt_vocab)
        # nên chia sẻ embedding + output projection: giảm tham số, thường cải thiện chất lượng dịch
        if tie_weights and src_vocab == tgt_vocab:
            self.src_emb.weight = self.tgt_emb.weight
            self.out_proj.weight = self.tgt_emb.weight

    def _init_weights(self, d_model, pad_id):
        for p in self.transformer.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
        for emb in (self.src_emb, self.tgt_emb):
            nn.init.normal_(emb.weight, mean=0.0, std=d_model ** -0.5)
            with torch.no_grad():
                emb.weight[pad_id].zero_()
        nn.init.normal_(self.out_proj.weight, mean=0.0, std=d_model ** -0.5)
        nn.init.zeros_(self.out_proj.bias)

    def make_src_key_padding_mask(self, src):
        return src == self.pad_id

    def make_tgt_mask(self, tgt):
        T = tgt.size(1)
        return nn.Transformer.generate_square_subsequent_mask(T, device=tgt.device)

    def encode(self, src, src_key_padding_mask):
        x = self.pos_enc(self.src_emb(src) * self.scale)
        return self.transformer.encoder(x, src_key_padding_mask=src_key_padding_mask)

    def decode(self, tgt, enc_out, tgt_mask, src_key_padding_mask, tgt_key_padding_mask=None):
        x = self.pos_enc(self.tgt_emb(tgt) * self.scale)
        return self.transformer.decoder(
            x, enc_out, 
            tgt_mask=tgt_mask, 
            memory_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask
        )

    def forward(self, src, tgt):
        src_key_padding_mask = self.make_src_key_padding_mask(src)
        tgt_key_padding_mask = self.make_src_key_padding_mask(tgt)
        tgt_mask = self.make_tgt_mask(tgt)

        enc_out = self.encode(src, src_key_padding_mask)
        dec_out = self.decode(tgt, enc_out, tgt_mask, src_key_padding_mask, tgt_key_padding_mask) 
        return self.out_proj(dec_out)

In [6]:
import math, random, torch
from torch.utils.data import Dataset, DataLoader

MAX_LEN = 128
MAX_TOKENS = 4096     # gom batch theo TỔNG SỐ TOKEN thay vì số câu -> VRAM/tốc độ ổn định hơn
                       # ~tương đương batch 128 câu ở độ dài trung bình cũ, chỉnh theo OOM/underutilization

def encode_pair(ex):
    src_ids = [BOS] + sp.encode(ex["vi"], out_type=int)[:MAX_LEN-2] + [EOS]
    tgt_ids = [BOS] + sp.encode(ex["en"], out_type=int)[:MAX_LEN-2] + [EOS]
    return {"src": src_ids, "tgt": tgt_ids}

class PhoMTDataset(Dataset):
    def __init__(self, hf_split):
        self.data = hf_split
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return encode_pair(self.data[idx])

def collate_fn(batch):
    src_max = max(len(x["src"]) for x in batch)
    tgt_max = max(len(x["tgt"]) for x in batch)
    src = torch.full((len(batch), src_max), PAD, dtype=torch.long)
    tgt = torch.full((len(batch), tgt_max), PAD, dtype=torch.long)
    for i, x in enumerate(batch):
        src[i, :len(x["src"])] = torch.tensor(x["src"])
        tgt[i, :len(x["tgt"])] = torch.tensor(x["tgt"])
    return src, tgt

def encode_all(vis, ens, tok_obj, bs=20000):
    S, T, dropped = [], [], 0
    for i in range(0, len(vis), bs):
        sv = tok_obj.encode_batch(vis[i:i + bs])
        te = tok_obj.encode_batch(ens[i:i + bs])
        for a, b in zip(sv, te):
            if 0 < len(a.ids) <= MAX_LEN - 2 and 0 < len(b.ids) <= MAX_LEN - 2:
                S.append([BOS] + a.ids + [EOS]); T.append([BOS] + b.ids + [EOS])
            else:
                dropped += 1
    print(f"encode xong: giữ {len(S)}, bỏ {dropped} cặp quá dài")
    return S, T

class PreEncoded(Dataset):
    def __init__(self, src, tgt):
        self.src, self.tgt = src, tgt
    def __len__(self):
        return len(self.src)
    def __getitem__(self, i):
        return {"src": self.src[i], "tgt": self.tgt[i]}

class TokenBucketBatchSampler:
    """Gom batch theo TỔNG SỐ TOKEN (không phải số câu). Vẫn bucket-sort trong từng
    cụm để câu dài gần nhau -> ít padding hơn."""
    def __init__(self, src, tgt, max_tokens, sort_chunk=20000, shuffle=True, seed=0):
        self.lens = [max(len(a), len(b)) for a, b in zip(src, tgt)]
        self.max_tokens, self.sort_chunk = max_tokens, sort_chunk
        self.shuffle, self.seed, self.epoch = shuffle, seed, 0

    def _pack(self, idx_sorted):
        batches, cur, cur_max = [], [], 0
        for i in idx_sorted:
            L = self.lens[i]
            new_max = max(cur_max, L)
            if cur and new_max * (len(cur) + 1) > self.max_tokens:
                batches.append(cur)
                cur, cur_max = [i], L
            else:
                cur.append(i); cur_max = new_max
        if cur:
            batches.append(cur)
        return batches

    def __iter__(self):
        rng = random.Random(self.seed + self.epoch); self.epoch += 1
        idx = list(range(len(self.lens)))
        if self.shuffle:
            rng.shuffle(idx)
        batches = []
        for i in range(0, len(idx), self.sort_chunk):
            c = sorted(idx[i:i + self.sort_chunk], key=self.lens.__getitem__)
            batches += self._pack(c)
        if self.shuffle:
            rng.shuffle(batches)
        return iter(batches)

    def __len__(self):
        return max(1, sum(self.lens) // self.max_tokens)

tr_src, tr_tgt = encode_all(train_vi, train_en, tokenizer_train)   # dùng bản có BPE-dropout
va_src, va_tgt = encode_all(valid_vi, valid_en, tokenizer)         # bản deterministic

train_loader = DataLoader(PreEncoded(tr_src, tr_tgt),
                          batch_sampler=TokenBucketBatchSampler(tr_src, tr_tgt, MAX_TOKENS, shuffle=True),
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
valid_loader = DataLoader(PreEncoded(va_src, va_tgt),
                          batch_sampler=TokenBucketBatchSampler(va_src, va_tgt, MAX_TOKENS, shuffle=False),
                          collate_fn=collate_fn, num_workers=2)
print("~steps/epoch:", len(train_loader))

encode xong: giữ 599957, bỏ 43 cặp quá dài
encode xong: giữ 5000, bỏ 0 cặp quá dài
~steps/epoch: 3110


In [7]:
import time

PEAK_LR = 5e-4
WARMUP = 4000
ACCUM_STEPS = 2     # gộp gradient 2 micro-batch trước khi update -> bù lại vì batch/VRAM đã giảm

def set_lr(optimizer, step, warmup=WARMUP, peak=PEAK_LR):
    step = max(step, 1)
    lr = peak * min(step / warmup, (warmup / step) ** 0.5)
    for g in optimizer.param_groups:
        g['lr'] = lr
    return lr

def make_optimizer(model):
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        if p.dim() < 2 or "emb" in name:
            no_decay.append(p)
        else:
            decay.append(p)
    return torch.optim.AdamW(
        [{"params": decay, "weight_decay": 0.01},
         {"params": no_decay, "weight_decay": 0.0}],
        lr=1e-7, betas=(0.9, 0.98), eps=1e-8,
    )

criterion = torch.nn.CrossEntropyLoss(ignore_index=PAD, label_smoothing=0.1)

def run_batch(model, src, tgt, device):
    src, tgt = src.to(device), tgt.to(device)
    tgt_in, tgt_out = tgt[:, :-1], tgt[:, 1:]
    logits = model(src, tgt_in)
    loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
    return loss

In [8]:
import time, math, torch, torch.nn as nn, torch.nn.functional as F

EPOCHS = 30
CKPT_EVERY = 5
CKPT_DIR = "/kaggle/working/ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_PATH = f"{CKPT_DIR}/last.pt"
BEST_PATH = f"{CKPT_DIR}/best.pt"
device = torch.device("cuda")

model = Transformer(VOCAB_SIZE, VOCAB_SIZE, pad_id=PAD).to(device)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
raw_model = model.module if isinstance(model, nn.DataParallel) else model
print(f"GPU: {torch.cuda.device_count()} | params: {sum(p.numel() for p in raw_model.parameters())/1e6:.1f}M")

optimizer = make_optimizer(model)
scaler = torch.amp.GradScaler("cuda")

@torch.no_grad()
def valid_nll():
    model.eval()
    tot, n = 0.0, 0
    for src, tgt in valid_loader:
        src, tgt = src.to(device), tgt.to(device)
        with torch.amp.autocast("cuda"):
            logits = model(src, tgt[:, :-1])
        out = tgt[:, 1:]
        tot += F.cross_entropy(logits.float().reshape(-1, logits.size(-1)), out.reshape(-1),
                               ignore_index=PAD, reduction="sum").item()
        n += (out != PAD).sum().item()
    model.train()
    return tot / n

def save_ckpt(path, epoch, step, vnll):
    ck = {"model": raw_model.state_dict(), "step": step, "epoch": epoch, "valid_nll": vnll}
    torch.save(ck, path)

def do_optimizer_step(step):
    lr = set_lr(optimizer, step)
    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    skipped = not torch.isfinite(grad_norm)
    if not skipped:
        scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)
    return lr, skipped

step, t_start = 0, time.time()
best_vnll = float("inf")
skipped_nan = 0
model.train()
optimizer.zero_grad(set_to_none=True)

for epoch in range(1, EPOCHS + 1):
    t_ep, run_loss, run_n = time.time(), 0.0, 0
    n_batches = len(train_loader)
    for micro_i, (src, tgt) in enumerate(train_loader):
        with torch.amp.autocast("cuda"):
            loss = run_batch(model, src, tgt, device) / ACCUM_STEPS

        if not torch.isfinite(loss):
            skipped_nan += 1
            print(f"[CẢNH BÁO] loss không hữu hạn ở epoch {epoch} micro-batch {micro_i} -> bỏ qua (tổng: {skipped_nan})")
            optimizer.zero_grad(set_to_none=True)
            continue

        scaler.scale(loss).backward()
        run_loss += loss.item() * ACCUM_STEPS; run_n += 1

        is_last_in_epoch = (micro_i == n_batches - 1)
        if (micro_i + 1) % ACCUM_STEPS == 0 or is_last_in_epoch:
            step += 1
            lr, skipped = do_optimizer_step(step)
            if skipped:
                skipped_nan += 1
                print(f"[CẢNH BÁO] grad không hữu hạn ở step {step} -> bỏ qua update (tổng: {skipped_nan})")
            if step % 200 == 0:
                print(f"ep {epoch} step {step} loss {run_loss/run_n:.4f} lr {lr:.2e}")
                run_loss, run_n = 0.0, 0

    vnll = valid_nll()
    print(f"=== epoch {epoch}/{EPOCHS} | valid NLL {vnll:.4f} (ppl {math.exp(vnll):.2f}) | "
          f"{(time.time()-t_ep)/60:.1f} phút/epoch | tổng {(time.time()-t_start)/3600:.2f}h | bỏ qua NaN: {skipped_nan} ===")

    if epoch % CKPT_EVERY == 0 or epoch == EPOCHS:
        save_ckpt(CKPT_PATH, epoch, step, vnll)
        push_files_to_kaggle([CKPT_PATH, TOKENIZER_PATH], message=f"epoch {epoch} valid_nll {vnll:.4f}")

    if vnll < best_vnll:
        best_vnll = vnll
        save_ckpt(BEST_PATH, epoch, step, vnll)
        push_files_to_kaggle([BEST_PATH], message=f"BEST epoch {epoch} valid_nll {vnll:.4f}")
        print(f"  -> best mới, đã lưu + đẩy best.pt (valid_nll={vnll:.4f})")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


GPU: 2 | params: 56.5M


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


ep 1 step 200 loss 9.2698 lr 2.50e-05
ep 1 step 400 loss 7.7004 lr 5.00e-05
ep 1 step 600 loss 7.1457 lr 7.50e-05
ep 1 step 800 loss 6.7439 lr 1.00e-04
ep 1 step 1000 loss 6.3949 lr 1.25e-04
ep 1 step 1200 loss 6.2299 lr 1.50e-04
ep 1 step 1400 loss 6.0486 lr 1.75e-04
=== epoch 1/30 | valid NLL 5.3255 (ppl 205.51) | 9.0 phút/epoch | tổng 0.15h | bỏ qua NaN: 0 ===
TẠO MỚI dataset checkpoint: Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/maihongsn/transformer-phomt-600k-ckpt
 
  -> best mới, đã lưu + đẩy best.pt (valid_nll=5.3255)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


...

ep 24 step 36600 loss 2.9118 lr 1.65e-04
ep 24 step 36800 loss 2.9343 lr 1.65e-04
ep 24 step 37000 loss 2.9538 lr 1.64e-04
ep 24 step 37200 loss 2.9533 lr 1.64e-04
[CẢNH BÁO] grad không hữu hạn ở step 37283 -> bỏ qua update (tổng: 10)
[CẢNH BÁO] grad không hữu hạn ở step 37378 -> bỏ qua update (tổng: 11)
ep 24 step 37400 loss 2.9704 lr 1.64e-04
ep 24 step 37600 loss 2.9791 lr 1.63e-04
ep 24 step 37800 loss 3.0002 lr 1.63e-04
ep 24 step 38000 loss 3.0024 lr 1.62e-04
=== epoch 24/30 | valid NLL 2.0994 (ppl 8.16) | 9.0 phút/epoch | tổng 3.65h | bỏ qua NaN: 11 ===
ep 25 step 38200 loss 2.9114 lr 1.62e-04
ep 25 step 38400 loss 2.9197 lr 1.61e-04
ep 25 step 38600 loss 2.9458 lr 1.61e-04
ep 25 step 38800 loss 2.9461 lr 1.61e-04
ep 25 step 39000 loss 2.9618 lr 1.60e-04
ep 25 step 39200 loss 2.9673 lr 1.60e-04
ep 25 step 39400 loss 2.9625 lr 1.59e-04
ep 25 step 39600 loss 2.9746 lr 1.59e-04
[CẢNH BÁO] grad không hữu hạn ở step 39709 -> bỏ qua update (tổng: 12)
=== epoch 25/30 | valid NLL 2.1032

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


ep 26 step 39800 loss 2.8825 lr 1.59e-04
ep 26 step 40000 loss 2.9053 lr 1.58e-04
ep 26 step 40200 loss 2.9187 lr 1.58e-04
ep 26 step 40400 loss 2.9318 lr 1.57e-04
ep 26 step 40600 loss 2.9520 lr 1.57e-04
ep 26 step 40800 loss 2.9583 lr 1.57e-04
ep 26 step 41000 loss 2.9448 lr 1.56e-04
ep 26 step 41200 loss 2.9770 lr 1.56e-04
=== epoch 26/30 | valid NLL 2.0882 (ppl 8.07) | 8.9 phút/epoch | tổng 3.96h | bỏ qua NaN: 12 ===
CẬP NHẬT checkpoint: Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/maihongsn/transformer-phomt-600k-ckpt
 
  -> best mới, đã lưu + đẩy best.pt (valid_nll=2.0882)


/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


ep 27 step 41400 loss 2.8601 lr 1.55e-04
ep 27 step 41600 loss 2.8895 lr 1.55e-04
[CẢNH BÁO] grad không hữu hạn ở step 41767 -> bỏ qua update (tổng: 13)
ep 27 step 41800 loss 2.9144 lr 1.55e-04
ep 27 step 42000 loss 2.9238 lr 1.54e-04
ep 27 step 42200 loss 2.9275 lr 1.54e-04
ep 27 step 42400 loss 2.9413 lr 1.54e-04
ep 27 step 42600 loss 2.9509 lr 1.53e-04
ep 27 step 42800 loss 2.9531 lr 1.53e-04
=== epoch 27/30 | valid NLL 2.1022 (ppl 8.18) | 8.9 phút/epoch | tổng 4.11h | bỏ qua NaN: 13 ===
ep 28 step 43000 loss 2.8647 lr 1.52e-04
ep 28 step 43200 loss 2.8784 lr 1.52e-04
ep 28 step 43400 loss 2.8949 lr 1.52e-04
ep 28 step 43600 loss 2.9093 lr 1.51e-04
ep 28 step 43800 loss 2.9219 lr 1.51e-04
ep 28 step 44000 loss 2.9266 lr 1.51e-04
ep 28 step 44200 loss 2.9375 lr 1.50e-04
[CẢNH BÁO] grad không hữu hạn ở step 44287 -> bỏ qua update (tổng: 14)
ep 28 step 44400 loss 2.9438 lr 1.50e-04
=== epoch 28/30 | valid NLL 2.1031 (ppl 8.19) | 8.9 phút/epoch | tổng 4.26h | bỏ qua NaN: 14 ===
ep 29 st

## Đánh giá / dịch thử (chạy sau khi train xong)

In [9]:
import os, glob, shutil, torch

DATASET_SLUG = "transformer-phomt-600k-ckpt"
KAGGLE_USERNAME = "maihongsn"
CKPT_DIR = "/kaggle/working/ckpt"
CKPT_NAME = "last.pt"
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_PATH = f"{CKPT_DIR}/{CKPT_NAME}"

if not os.path.exists(CKPT_PATH):
    candidates = glob.glob(f"/kaggle/input/**/{DATASET_SLUG}/{CKPT_NAME}", recursive=True)
    if not candidates:
        raise FileNotFoundError(f"Không thấy {CKPT_NAME}. Hãy Add Input dataset {KAGGLE_USERNAME}/{DATASET_SLUG}")
    shutil.copy(candidates[0], CKPT_PATH)
    print("Đã copy checkpoint từ:", candidates[0])

device = torch.device("cuda")
ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
state = {(k[7:] if k.startswith("module.") else k): v for k, v in ck["model"].items()}

model = Transformer(VOCAB_SIZE, VOCAB_SIZE, pad_id=PAD)
model.load_state_dict(state)
model.to(device).eval()

print(f"Đã nạp {CKPT_NAME}: step={ck['step']}, epoch={ck['epoch']}, "
      f"params={sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
del ck, state

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = TransformerEncoder(


Đã nạp last.pt: step=47669, epoch=30, params=56.5M


In [10]:
# đo NLL thật (không label smoothing, không dropout)
import math
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset

def get_split(raw, names):
    for n in names:
        if n in raw:
            return raw[n]
    raise KeyError(f"Không thấy split nào trong {names}. Hiện có: {list(raw.keys())}")

@torch.no_grad()
def eval_nll(hf_split, n_samples=30000, batch_size=128, seed=0):
    model.eval()
    ds = PhoMTDataset(hf_split)
    g = torch.Generator().manual_seed(seed)
    idx = torch.randperm(len(ds), generator=g)[:min(n_samples, len(ds))].tolist()
    dl = DataLoader(Subset(ds, idx), batch_size=batch_size, shuffle=False,
                    collate_fn=collate_fn, num_workers=2)
    tot_nll, tot_tok = 0.0, 0
    for src, tgt in dl:
        src, tgt = src.to(device), tgt.to(device)
        with torch.amp.autocast('cuda'):
            logits = model(src, tgt[:, :-1])
        tgt_out = tgt[:, 1:]
        nll = F.cross_entropy(logits.float().reshape(-1, logits.size(-1)),
                      tgt_out.reshape(-1), ignore_index=PAD, reduction='sum')
        tot_nll += nll.item()
        tot_tok += (tgt_out != PAD).sum().item()
    return tot_nll / tot_tok

valid_split = get_split(raw, ["validation", "valid", "dev"])
train_nll = eval_nll(train_sub)
valid_nll = eval_nll(valid_split)
print(f"train NLL: {train_nll:.4f}  (ppl {math.exp(train_nll):.2f})")
print(f"valid NLL: {valid_nll:.4f}  (ppl {math.exp(valid_nll):.2f})")
print(f"gap valid - train: {valid_nll - train_nll:+.4f}")

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


TypeError: Caught TypeError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 52, in fetch
    data = self.dataset.__getitems__(possibly_batched_index)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataset.py", line 413, in __getitems__
    return [self.dataset[self.indices[idx]] for idx in indices]
            ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_23/1604250601.py", line 19, in __getitem__
    return encode_pair(self.data[idx])
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_23/1604250601.py", line 9, in encode_pair
    src_ids = [BOS] + sp.encode(ex["vi"], 

In [ ]:
def clean(split):
    vis, ens = split["vi"], split["en"]
    good = [i for i, (v, e) in enumerate(zip(vis, ens))
            if isinstance(v, str) and isinstance(e, str)]
    print(f"giữ {len(good)}/{len(split)} dòng")
    return split.select(good, keep_in_memory=True)   # không ghi cache ra /kaggle/input

valid_split = clean(get_split(raw, ["validation", "valid", "dev"]))

if "train_nll" not in globals():                      # phòng khi kernel đã restart
    train_nll = eval_nll(train_sub)
valid_nll = eval_nll(valid_split)

print(f"train NLL: {train_nll:.4f}  (ppl {math.exp(train_nll):.2f})")
print(f"valid NLL: {valid_nll:.4f}  (ppl {math.exp(valid_nll):.2f})")
print(f"gap valid - train: {valid_nll - train_nll:+.4f}")

In [ ]:
# beam search theo batch
@torch.no_grad()
def beam_search(model, src, beam=4, max_len=128, alpha=0.6):
    model.eval()
    B, dev = src.size(0), src.device
    with torch.amp.autocast('cuda'):
        src_key_padding_mask = model.make_src_key_padding_mask(src)
        enc = model.encode(src, src_key_padding_mask)
    
    enc = enc.repeat_interleave(beam, 0)
    src_key_padding_mask = src_key_padding_mask.repeat_interleave(beam, 0)

    seqs = torch.full((B * beam, 1), BOS, dtype=torch.long, device=dev)
    scores = torch.zeros(B, beam, device=dev)
    scores[:, 1:] = -1e9                      
    scores = scores.view(-1)
    finished = torch.zeros(B * beam, dtype=torch.bool, device=dev)
    base = (torch.arange(B, device=dev) * beam).unsqueeze(1)

    for _ in range(max_len):
        with torch.amp.autocast('cuda'):
            tgt_mask = model.make_tgt_mask(seqs)
            # Gọi hàm decode tương thích Torch
            dec = model.decode(seqs, enc, tgt_mask, src_key_padding_mask)[:, -1]
            logits = model.out_proj(dec)
        logp = F.log_softmax(logits.float(), dim=-1)      
        V = logp.size(-1)

        logp[finished] = -1e9
        logp[finished, PAD] = 0

        cand = (scores.unsqueeze(1) + logp).view(B, beam * V)
        top_scores, top_idx = cand.topk(beam, dim=1)
        beam_idx = top_idx // V
        tok = (top_idx % V).view(-1)
        sel = (base + beam_idx).view(-1)

        seqs = torch.cat([seqs[sel], tok.unsqueeze(1)], dim=1)
        scores = top_scores.view(-1)
        finished = finished[sel] | (tok == EOS)
        if finished.all():
            break

    seqs = seqs.view(B, beam, -1)[:, :, 1:]               
    scores = scores.view(B, beam)
    best = []
    for b in range(B):
        cands = []
        for k in range(beam):
            ids = seqs[b, k].tolist()
            if EOS in ids:
                ids = ids[:ids.index(EOS)]
            ids = [i for i in ids if i != PAD]
            lp = ((5 + len(ids) + 1) / 6) ** alpha        
            cands.append((scores[b, k].item() / lp, ids))
        best.append(max(cands, key=lambda x: x[0])[1])
    return best

@torch.no_grad()
def translate(texts, beam=4, batch_size=32, max_len=128):
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))   
    results = [None] * len(texts)
    for s in range(0, len(order), batch_size):
        idxs = order[s:s + batch_size]
        batch = [[BOS] + sp.encode(texts[i])[:MAX_LEN - 2] + [EOS] for i in idxs]
        L = max(len(b) for b in batch)
        src = torch.full((len(batch), L), PAD, dtype=torch.long)
        for j, b in enumerate(batch):
            src[j, :len(b)] = torch.tensor(b)
        out_ids = beam_search(model, src.to(device), beam=beam, max_len=max_len)
        for i, ids in zip(idxs, out_ids):
            results[i] = sp.decode(ids)
    return results

In [ ]:
#sacreBLEU trên tập test (vi -> en)
import subprocess
subprocess.run(["pip", "install", "-q", "sacrebleu"])
import sacrebleu, time

test_split = get_split(raw, ["test"])
N = 2000                                   # tăng lên nếu muốn đo đầy đủ hơn (chậm hơn)
part = test_split[:N]
srcs, refs = part["vi"], part["en"]

for beam in (1, 4):                        # 1 = greedy
    t0 = time.time()
    hyps = translate(srcs, beam=beam)
    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    print(f"beam={beam}: {bleu}  ({time.time() - t0:.0f}s)")

print("\n--- Câu mẫu (beam=4) ---")
for i in range(0, 10):
    print("VI :", srcs[i])
    print("REF:", refs[i])
    print("HYP:", hyps[i])
    print()

In [ ]:
s = "Sadly, Brother Albert Barnett and his wife, Susan, were killed."
enc = tokenizer.encode(s)
print("tokens :", enc.tokens)
print("decode :", repr(tokenizer.decode(enc.ids)))
print("pre_tokenizer:", tokenizer.pre_tokenizer)
print("decoder      :", tokenizer.decoder)
print("normalizer   :", tokenizer.normalizer)